### Introduction

In this notebook you will be learning how to use the shallow water model provided. Recall that the shallow water equations are, for height variable $h$, horizontal speed $u, v$, in non-dimensional form and in Cartesian coordinates $(x,y)$, are:

$$
\begin{align}
\partial_t h + \partial_x(hu) + \partial_y(hv) &= F\\
\partial_t u + u\partial_x u + v\partial_y v - \text{Ro}^{-1}v + \partial_x h &= 0\\
\partial_t v + u\partial_x v + v\partial_y v + \text{Ro}^{-1}u + \partial_y h &= 0
\end{align}
$$

where Ro is the Rossby number and $F$ is any forcing function (we will ignore this for now)

### How the model works
The model is run from `driver.py`. Here the grid is initialised along with input/output and the timestepping functions. The algorithm for solving the equations is as follows.

The three equations above are written in the form:
$$
\begin{align}
\partial_t h &= A + F,\\
\partial_t u &=  B,\\
\partial_t v & = C.
\end{align}
$$
Following Arakawa and Lamb 1980, methods in the `ArakawaCGrid` class discretise the spatial parts of the PDEs, $A, B, C$ on the Arakawa C grid, using an algorithm that preserves global energy and enstrophy conservation. 

The three "tendencies", i.e. $\partial_t h, \partial_t u, \partial_t v$ as calculated above are then passed to the timestepping module, that then implements a Adam Bashford method that advances the three variables, $h, u, v$, to the next timestep.

The forcing function $F$ can be specified by the user, as well as the initial condition.

### Model grid
In the non-dimensionalised coordinates, the default model grid is a 2*N by N grid in $x$ and $y$ direction spanning $x=(-1,1)$ and $y=(-0.5,0.5)$, somewhat mimicking the Earth's longitude and latitude coordinates respectively. 

        # j+1    q-----v-----q-----v-----q
        #        |           |           |
        # j+1/2  u     h     u     h     u
        #        |           |           |
        # j      q-----v-----q-----v-----q
        #        |           |           |
        # j-1/2  u     h     u     h     u
        #        |           |           |
        # j-1    q-----v-----q-----v-----q
        #
        #        i   i-1/2.  i   i+1/2  i+1

Recall that the variables are defined on the Arakawa C-grid, as shown above (where $q$ is the vorticity). Care should therefore be taken when definining initial conditions to ensure that array sizes are equal.

### Mounting google drive and installing
If you downloaded the code in your google drive, then run the following:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

! pip install -e /content/drive/MyDrive/SW_summerschool/

In [ ]:
import sys
import importlib
sys.modules["imp"] = importlib
from google.colab import output
output.enable_custom_widget_manager()

### Now restart kernel using Runtime/Restart session in the colab menu

The config/output files for this exercise should be found/put in the directory `/content/drive/MyDrive/SW_summerschool/examples/Introduction`

In [ ]:
# Need to change into directory with config files
%cd /content/drive/MyDrive/SW_summerschool/examples/Introduction

### If running the python notebook locally, run from here

In [ ]:
# First load the necessary modules.
%load_ext autoreload
%autoreload 2
%matplotlib widget
from IPython.display import HTML
import xarray as xr
import numpy as np
from sw_summerschool.plotting import animate_height_velocity, animate_contour
from sw_summerschool import SW_model
from sw_summerschool.helper import interp_to_centre
import matplotlib as mpl
import matplotlib.pyplot as plt


### Model initialisation
We will initialise a model object below. To do so, we must give the initialising function a config file `cfg.yaml` and output file name (of netcdf form, e.g., `outfile.nc`). Parameters found in the config `yaml` file can be overridden by passing the variable straight to the initialising function

### Implementing the initial condition

We haven't yet defined the `gaussian` initial condition function referred to in the configuration. To test the model, let's create an initial condition on the $h$ variable:
$$
h_0 = 1. + \delta \exp\left(-\frac{(x-x_0)^2 + (y-y_0)^2}{2\sigma^2}\right)
$$
where $\delta$ is the amplitude, $\sigma$ the width and $x_0$ and $y_0$ the position of the centre of the Gaussian. We will leave the velocities $u$, $v$ unchanged. To implement this condition, go to the file `/content/SW_summerschool/src/sw_summerschool/initial_conditions.py` and alter the `gaussian` function. The function takes as its input the

Changes should be made to `initial_conditions.py`


In [ ]:
# Create a model with default settings, with large Rossby number, U/fL (i.e., rotation very weak)
model = SW_model("cfg.yaml", outfile = 'intro.nc', io_freq = 100, print_freq =100, Ro=0.1, initial_condition='vortex')

### Integrating the model

Now integrate the model by $N_t$ timesteps. We can do this however many times we want, with output being saved to the `outfile` at every `io_freq` timesteps. 

In [ ]:
N_t = 10000
model.integrate(N_t)

### Plotting the output

To view the output, we have to manipulate the output such that the variables are all on the same grid. Recall that currently, $u$ and $v$ are defined on staggered grids offset from the grid centres, where $h$ is defined.

Here I will use the package `xarray` to open and manipulate the netcdf output file.

In [ ]:
ds = xr.open_dataset('intro.nc')
ds

In [ ]:

plt.close('all')

In [ ]:
# Interpolate u and v to grid centres using helper function
u = interp_to_centre("u", ds, bc="periodic")
v = interp_to_centre("v", ds, bc="periodic")
q = interp_to_centre("q", ds, bc="periodic")

# Plotting functions take np arrays, so convert xarray dataarray into numpy array
h = ds["h"].data
t = ds["time"].data
x = ds["xmid"].data
y = ds["ymid"].data

In [ ]:
# Use the plotting functions defined in plotting.py
from sw_summerschool.plotting import plot_height_velocity_snapshot, animate_height_velocity, plot_contour_snapshot

In [ ]:
plot_height_velocity_snapshot(h, u, v, t, x, y, index=-1, scale="snapshot", quiver_skip=2)

# Look at the vorticity, q
plot_contour_snapshot(q, t, x, y, index=-1)

# Now let's animate the same result

anim = animate_height_velocity(h, u, v, t, x, y, quiver_skip=2, quiver_scale=0.1, interval=100)

In [ ]:
# Run this cell to close the 
plt.close('all')

In [ ]:
# Create "player"
anim_html = HTML(anim.to_jshtml())

In [ ]:
# View player
anim_html

### Further Tasks
Now that you have managed to perform your first run with the shallow water model, have a play around with the settings and configurations. Here are some suggested trials:

* Try integrating the model with increasingly larger timesteps. Plot how the total energy of the model changes over time in these simulations
* Include the effect of rotation by decreasing the Rossby number, Ro. What changes as you make Ro<<1?
* Try the "vortex" initial condition. Describe what happens to the flow as the rotation rate is increased (Ro-->0). Note at Ro=0 the simulation will crash (divide by zero errors, associated rotation rate tends to infinity).
* Try the "free-slip-wall" y boundary condition. What happens to the flow at the y boundary in comparison to the periodic case?